# CityLens Baseline Comparison

Use this notebook for the next experiment phase only:

- lock the current `prithvi_rgb_lora` reference results
- run fair satellite backbone comparisons
- summarize metrics into one table
- decide whether street-only and fusion are worth running next

This notebook is intentionally smaller than `colab_citylens_full.ipynb` and calls the existing training entrypoints instead of duplicating the full pipeline.

## Colab setup

Upload this notebook to a fresh Colab runtime, choose a GPU, then run all cells. This notebook mounts Drive, clones the repo, installs the learned-model dependencies, and runs only the comparison-phase experiments.

In [37]:
!pip install -q virtualenv
!rm -rf /content/citylens_venv
!virtualenv /content/citylens_venv

!/content/citylens_venv/bin/python -m pip install -U pip wheel
!/content/citylens_venv/bin/python -m pip install --no-cache-dir \
  "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.5.2" "pandas==2.2.2"

created virtual environment CPython3.12.12.final.0-64-x86_64 in 368ms
  creator CPython3Posix(dest=/content/citylens_venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
Using cached packaging-26.0-py3-none-any.whl (74 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [wheel]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 190.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 274.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 275.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 221.9 MB/

In [38]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path('/content/CityLens')
REPO_URL = 'https://github.com/wojackbro/GEOBENCH.git'

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
else:
    if (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'fetch', 'origin'], cwd=str(REPO_ROOT), check=True)
        subprocess.run(['git', 'checkout', 'main'], cwd=str(REPO_ROOT), check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=str(REPO_ROOT), check=True)
    else:
        raise SystemExit(f'Expected a git repo at {REPO_ROOT}, but .git is missing.')

# Auto-detect project root (some Colab runs have nested /content/CityLens/CityLens)
CANDIDATE_ROOTS = [REPO_ROOT, REPO_ROOT / 'CityLens']
PROJECT_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / 'evaluate' / 'global_learned' / 'train.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise SystemExit('Could not locate evaluate/global_learned/train.py after clone/pull.')

commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(REPO_ROOT), text=True).strip()
print('Repo root:', REPO_ROOT)
print('Project root:', PROJECT_ROOT)
print('Commit:', commit)

Repo root: /content/CityLens
Project root: /content/CityLens/CityLens
Commit: 319729e


In [39]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data')
os.environ['CITYLENS_DATA_ROOT'] = str(DATA_ROOT)
print('CITYLENS_DATA_ROOT =', os.environ['CITYLENS_DATA_ROOT'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CITYLENS_DATA_ROOT = /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data


In [40]:
required = [
    DATA_ROOT / 'Benchmark',
    DATA_ROOT / 'Results',
    PROJECT_ROOT,
    PROJECT_ROOT / 'evaluate' / 'global_learned' / 'train.py',
]
for path in required:
    print(path, 'exists=' + str(path.exists()))

if not (DATA_ROOT / 'Benchmark').exists():
    raise SystemExit('CityLens data not found in Drive. Reuse the previous full notebook once to prepare the dataset first.')
if not (PROJECT_ROOT / 'evaluate' / 'global_learned' / 'train.py').exists():
    raise SystemExit('Project checkout is missing evaluate/global_learned/train.py. Re-run the repo clone/pull cell.')

/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Benchmark exists=True
/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results exists=True
/content/CityLens/CityLens exists=True
/content/CityLens/CityLens/evaluate/global_learned/train.py exists=True


In [41]:
from pathlib import Path
import json
import os
import subprocess
import pandas as pd

DATA_ROOT = Path('/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data')
CITYLENS_ROOT = PROJECT_ROOT
RESULTS_ROOT = DATA_ROOT / 'Results' / 'global_learned'

REFERENCE_MODELS = ['prithvi_rgb_lora', 'dinov2_sat', 'resnet50_sat']
TASK_EPOCHS = {
    'gdp': 20,
    'acc2health': 30,
    'build_height': 30,
    'pop': 5,
}

LOCKED_PRITHVI = {
    'gdp': {'r2': 0.5807730555534363, 'rmse': 331463584.0, 'best_epoch': 14, 'epochs': 20},
    'acc2health': {'r2': 0.39014101028442383, 'rmse': 9.55018424987793, 'best_epoch': 9, 'epochs': 30},
    'build_height': {'r2': 0.8681523203849792, 'rmse': 2.534539222717285, 'best_epoch': 9, 'epochs': 30},
    'pop': {'r2': -0.03239285945892334, 'rmse': 21641.24609375, 'best_epoch': 2, 'epochs': 5},
}

COMMON = {
    'branch': 'satellite',
    'street_model': 'clip_vitb16',
    'batch_size': 8,
    'image_size': 224,
    'lr': 2e-4,
    'backbone_lr': 2e-4,
    'head_lr': 1e-3,
    'weight_decay': 1e-2,
    'seed': 42,
    'lora_r': 8,
    'target_transform': 'log1p',
    'val_frac': 0.1,
    'num_workers': 2,
}

assert DATA_ROOT.exists(), f'Missing data root: {DATA_ROOT}'
assert CITYLENS_ROOT.exists(), f'Missing project root: {CITYLENS_ROOT}'
assert (CITYLENS_ROOT / 'evaluate' / 'global_learned' / 'train.py').exists(), 'Missing evaluate/global_learned/train.py in CITYLENS_ROOT'
print('Project root:', CITYLENS_ROOT)
print('Results root:', RESULTS_ROOT)

Project root: /content/CityLens/CityLens
Results root: /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned


In [42]:
import os
import sys


def run_train(
    task_name: str,
    branch: str,
    epochs: int,
    satellite_model: str = 'prithvi_rgb_lora',
    street_model: str = 'clip_vitb16',
    fusion_type: str = 'late',
    pooling: str = 'mean',
    resume: bool = True,
    skip_if_done: bool = True,
):
    # Build CLI args for argparse (run in-process so Colab shows the real traceback;
    # subprocess + sys.executable often hides errors or uses a different env than the kernel).
    cli = [
        '--task_name', task_name,
        '--branch', branch,
        '--satellite_model', satellite_model,
        '--street_model', street_model,
        '--fusion_type', fusion_type,
        '--pooling', pooling,
        '--epochs', str(epochs),
        '--batch_size', str(COMMON['batch_size']),
        '--image_size', str(COMMON['image_size']),
        '--lr', str(COMMON['lr']),
        '--backbone_lr', str(COMMON['backbone_lr']),
        '--head_lr', str(COMMON['head_lr']),
        '--weight_decay', str(COMMON['weight_decay']),
        '--seed', str(COMMON['seed']),
        '--lora_r', str(COMMON['lora_r']),
        '--target_transform', COMMON['target_transform'],
        '--val_frac', str(COMMON['val_frac']),
        '--num_workers', str(COMMON['num_workers']),
        '--data_root', str(DATA_ROOT),
    ]
    if resume:
        cli.append('--resume')
    if skip_if_done:
        cli.append('--skip_if_done')

    print('\n$ ' + ' '.join([sys.executable, '-m', 'evaluate.global_learned.train'] + cli))

    root = str(CITYLENS_ROOT)
    argv_bak = sys.argv[:]
    cwd_bak = os.getcwd()
    try:
        os.chdir(root)
        if root not in sys.path:
            sys.path.insert(0, root)
        sys.argv = ['evaluate.global_learned.train'] + cli
        from evaluate.global_learned.train import main as _train_main

        _train_main()
    finally:
        sys.argv = argv_bak
        os.chdir(cwd_bak)


def run_satellite_matrix(tasks, models):
    for task_name in tasks:
        epochs = TASK_EPOCHS[task_name]
        for satellite_model in models:
            run_train(task_name, branch='satellite', satellite_model=satellite_model, epochs=epochs)


def run_street_matrix(tasks, street_models):
    for task_name in tasks:
        epochs = TASK_EPOCHS[task_name]
        for street_model in street_models:
            run_train(task_name, branch='street', street_model=street_model, epochs=epochs)


def run_fusion_matrix(tasks, street_models, fusion_types=('late', 'gated'), satellite_model='prithvi_rgb_lora'):
    for task_name in tasks:
        epochs = TASK_EPOCHS[task_name]
        for street_model in street_models:
            for fusion_type in fusion_types:
                run_train(
                    task_name,
                    branch='fusion',
                    satellite_model=satellite_model,
                    street_model=street_model,
                    fusion_type=fusion_type,
                    epochs=epochs,
                )


## Run controls (satellite archived, street/fusion next)

Satellite baseline stage is complete.

Use the run-control cell to choose what runs now:

1. `RUN_SATELLITE=False` (default, avoid re-running completed baselines)
2. `RUN_STREET=True` to launch street-only matrix
3. `RUN_FUSION=True` after street results are ready

Keep `RUN_*` flags `False` for stages you do not want to execute.

In [43]:
from pathlib import Path
import subprocess

def run_street_matrix(tasks, street_models):
    for task_name in tasks:
        epochs = TASK_EPOCHS[task_name]
        for street_model in street_models:
            try:
                run_train(task_name, branch='street', street_model=street_model, epochs=epochs)
            except subprocess.CalledProcessError as e:
                print(f"[SKIP] street/{task_name}/{street_model} failed (likely no records).")

p = Path("/content/CityLens/CityLens/evaluate/global_learned/models.py")
txt = p.read_text()

old = 'self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)'
new = 'self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)'

if old in txt and new not in txt:
    txt = txt.replace(old, new)
    p.write_text(txt)
    print("Patched TimmEncoder to force img_size=224")
else:
    print("Patch already applied or line not found")

Patch already applied or line not found


In [44]:
!rg "create_model\(model_name, pretrained=True, num_classes=0, img_size=224\)" /content/CityLens/CityLens/evaluate/global_learned/models.py

39:            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)


In [45]:
from pathlib import Path

p = Path("/content/CityLens/CityLens/evaluate/global_learned/utils.py")
txt = p.read_text()

old = "rmse = mean_squared_error(y_true, y_pred, squared=False)"
new = "rmse = float(np.sqrt(mse))"

if old in txt:
    txt = txt.replace(old, new)
    p.write_text(txt)
    print("Patched compute_metrics: rmse now uses sqrt(mse)")
else:
    print("Patch target not found (may already be patched)")

Patch target not found (may already be patched)


In [46]:
!rg "img_size=224" /content/CityLens/CityLens/evaluate/global_learned/models.py

39:            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)


In [47]:
!rg "rmse = " /content/CityLens/CityLens/evaluate/global_learned/utils.py

93:    rmse = float(np.sqrt(mse))


In [48]:
from pathlib import Path

p = Path("/content/CityLens/CityLens/evaluate/global_learned/models.py")
txt = p.read_text()

old_block = """        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)
"""

new_block = """        # Only some timm models (e.g., ViTs) accept img_size in constructor.
        # ResNet and others can fail if img_size is passed.
        if "vit" in model_name or "dinov2" in model_name:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)
        else:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)
"""

if old_block in txt:
    txt = txt.replace(old_block, new_block)
    p.write_text(txt)
    print("Patched TimmEncoder: img_size only for ViT/DINO models.")
else:
    print("Expected block not found; print constructor section manually and patch by hand.")

Expected block not found; print constructor section manually and patch by hand.


In [49]:
!sed -n '28,50p' /content/CityLens/CityLens/evaluate/global_learned/models.py

        return out
    return out.view(out.size(0), -1)


class TimmEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.model_name = model_name
        # Only some timm models (e.g., ViTs) accept img_size in constructor.
        # ResNet and others can fail if img_size is passed.
        if "vit" in model_name or "dinov2" in model_name:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=224)
        else:
            self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        feat = reduce_backbone_output(out)
        if self.feature_dim is None:
            self.feature_dim = feat.size(-1)
        return feat



In [50]:
import sys, subprocess, shutil
from pathlib import Path

py = sys.executable

for _ in range(3):
    subprocess.call([py, "-m", "pip", "uninstall", "-y", "numpy", "scipy", "scikit-learn", "pandas"])

base = Path("/usr/local/lib/python3.12/dist-packages")
for p in list(base.glob("numpy")) + list(base.glob("numpy-*.dist-info")) + list(base.glob("numpy.libs")):
    print("rm", p)
    shutil.rmtree(p, ignore_errors=True)

subprocess.check_call([
    py, "-m", "pip", "install", "--no-cache-dir", "--no-deps",
    "numpy==1.26.4",
])
subprocess.check_call([
    py, "-m", "pip", "install", "--no-cache-dir",
    "scipy==1.13.1", "scikit-learn==1.5.2", "pandas==2.2.2",
])

0

In [51]:
# Run controls for targeted fusion completion
TASKS_TO_RUN = ['acc2health']   # skip gdp, acc2health now
SAT_MODELS_TO_RUN = ['resnet50_sat']  # ignored since RUN_SATELLITE=False

# Option A (most compact)
STREET_MODELS_TO_RUN = ['resnet50']
FUSION_TYPES_TO_RUN = ['gated']

RUN_SATELLITE = False
RUN_STREET = False
RUN_FUSION = False
if RUN_SATELLITE:
    run_satellite_matrix(TASKS_TO_RUN, SAT_MODELS_TO_RUN)

if RUN_STREET:
    run_street_matrix(TASKS_TO_RUN, STREET_MODELS_TO_RUN)

if RUN_FUSION:
    # Uses Prithvi satellite branch plus each selected street model.
    run_fusion_matrix(
        TASKS_TO_RUN,
        STREET_MODELS_TO_RUN,
        fusion_types=FUSION_TYPES_TO_RUN,
        satellite_model='prithvi_rgb_lora',
    )

if not any([RUN_SATELLITE, RUN_STREET, RUN_FUSION]):
    print('Dry run mode: all run switches are False.')
    print('Set RUN_STREET=True to start the next step.')


Dry run mode: all run switches are False.
Set RUN_STREET=True to start the next step.


In [52]:
!/content/citylens_venv/bin/python -m pip install -U pip
!/content/citylens_venv/bin/python -m pip install --no-cache-dir torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 530.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 560.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 520.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 530.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 606.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 618.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 537.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 611.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 673.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 496.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 101.0 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 506.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 557.2

In [53]:
!/content/citylens_venv/bin/python -m pip install --no-cache-dir \
  timm==1.0.9 peft==0.12.0 terratorch safetensors tqdm pillow pyyaml

INFO: pip is looking at multiple versions of terratorch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of terratorch to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 126.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 122.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.0/849.0 kB 58.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 513.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 173.8 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━

In [55]:
!pip install -q virtualenv
!rm -rf /content/citylens_venv
!virtualenv /content/citylens_venv
!/content/citylens_venv/bin/python -m pip install -U pip
!/content/citylens_venv/bin/python -m pip install --no-cache-dir numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.2 pandas==2.2.2
!/content/citylens_venv/bin/python -m pip install --no-cache-dir torch torchvision
!/content/citylens_venv/bin/python -m pip install --no-cache-dir timm==1.0.9 peft==0.12.0 terratorch safetensors tqdm pillow matplotlib

created virtual environment CPython3.12.12.final.0-64-x86_64 in 163ms
  creator CPython3Posix(dest=/content/citylens_venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 546.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 385.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 319.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 378.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 175.2 MB/s  0:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 164.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 103

In [56]:
!/content/citylens_venv/bin/python -c "import numpy as np; import numpy.rec; import sklearn; print('ok', np.__version__)"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'numpy.rec'


In [57]:
%pip uninstall -y numpy scipy scikit-learn pandas
%pip install --no-cache-dir --force-reinstall --ignore-installed numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.2 pandas==2.2.2

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 134.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 292.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 221.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 304.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 192.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 368.5 M

In [1]:
from pathlib import Path
import json
import pandas as pd

DATA_ROOT = Path('/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data')
RESULTS_ROOT = DATA_ROOT / 'Results' / 'global_learned'
TASKS = ['gdp', 'acc2health', 'build_height', 'pop']

def read_json(p):
    try:
        return json.loads(p.read_text())
    except Exception:
        return None

rows = []
for task in TASKS:
    task_dir = RESULTS_ROOT / task
    if not task_dir.exists():
        continue

    for exp_dir in task_dir.iterdir():
        if not exp_dir.is_dir():
            continue
        name = exp_dir.name
        if f"-{task}-" not in f"-{name}-":
            pass
        # focus on fusion runs
        if "-fusion-" not in name:
            continue

        cfg = read_json(exp_dir / "config.json") or {}
        metrics = read_json(exp_dir / "metrics.json")
        history_path = exp_dir / "history.csv"
        best_ckpt = exp_dir / "checkpoints" / "best.pt"
        last_ckpt = exp_dir / "checkpoints" / "last.pt"

        status = "not_started"
        if metrics and best_ckpt.exists():
            status = "finished_or_best_saved"
        elif last_ckpt.exists():
            status = "in_progress_or_interrupted"

        rows.append({
            "task": task,
            "exp_dir": name,
            "street_model": cfg.get("street_model"),
            "fusion_type": cfg.get("fusion_type"),
            "epochs_target": cfg.get("epochs"),
            "best_epoch": (metrics or {}).get("best_epoch"),
            "r2": (metrics or {}).get("r2"),
            "rmse": (metrics or {}).get("rmse"),
            "has_best_pt": best_ckpt.exists(),
            "has_last_pt": last_ckpt.exists(),
            "has_history": history_path.exists(),
            "status": status,
        })

df = pd.DataFrame(rows)
if df.empty:
    print("No fusion runs found yet.")
else:
    display(df.sort_values(["task","street_model","fusion_type"]).reset_index(drop=True))
    print("\nSummary by status:")
    display(df.groupby("status").size().rename("count").reset_index())

,task,exp_dir,street_model,fusion_type,epochs_target,best_epoch,r2,rmse,has_best_pt,has_last_pt,has_history,status
0,acc2health,acc2health-fusion-prithvi_rgb_lora-clip_vitb16...,clip_vitb16,gated,30.0,19.0,0.167151,9.988541e+00,False,False,True,not_started
1,acc2health,acc2health-fusion-prithvi_rgb_lora-clip_vitb16...,clip_vitb16,late,30.0,13.0,0.134735,1.018108e+01,False,False,True,not_started
2,acc2health,acc2health-fusion-prithvi_rgb_lora-resnet50-ga...,resnet50,gated,30.0,3.0,0.161168,1.002436e+01,False,True,True,in_progress_or_interrupted
3,acc2health,acc2health-fusion-prithvi_rgb_lora-resnet50-la...,resnet50,late,30.0,12.0,0.190886,9.845183e+00,False,False,True,not_started
4,acc2health,acc2health-fusion-prithvi_rgb_lora-dinov2_vitb...,None,None,NaN,NaN,NaN,NaN,False,False,False,not_started
5,build_height,build_height-fusion-prithvi_rgb_lora-dinov2_vi...,dinov2_vitb14,gated,30.0,21.0,0.865288,2.596345e+00,False,False,True,not_started
6,build_height,build_height-fusion-prithvi_rgb_lora-dinov2_vi...,dinov2_vitb14,late,30.0,23.0,0.872343,2.527444e+00,False,False,True,not_started
7,build_height,build_height-fusion-prithvi_rgb_lora-resnet50-...,resnet50,gated,30.0,28.0,0.859323,2.653199e+00,False,False,True,not_started
8,build_height,build_height-fusion-prithvi_rgb_lora-resnet50-...,resnet50,late,30.0,28.0,0.848417,2.754131e+00,False,False,True,not_started
9,gdp,gdp-fusion-prithvi_rgb_lora-clip_vitb16-gated-...,clip_vitb16,gated,20.0,18.0,0.015131,5.629208e+08,False,False,True,not_started



Summary by status:


,status,count
0,in_progress_or_interrupted,3
1,not_started,16


In [2]:
TASKS = ['gdp', 'acc2health', 'build_height', 'pop']
STREET_MODELS = ['clip_vitb16', 'resnet50', 'dinov2_vitb14']
FUSION_TYPES = ['late', 'gated']

done = set()
for _, r in df.iterrows():
    if r["has_best_pt"] and pd.notna(r["r2"]):
        done.add((r["task"], r["street_model"], r["fusion_type"]))

todo = []
for t in TASKS:
    for s in STREET_MODELS:
        for f in FUSION_TYPES:
            if (t, s, f) not in done:
                todo.append((t, s, f))

print(f"Done: {len(done)} | Missing: {len(todo)}")
for x in todo[:50]:
    print("MISSING:", x)

Done: 0 | Missing: 24
MISSING: ('gdp', 'clip_vitb16', 'late')
MISSING: ('gdp', 'clip_vitb16', 'gated')
MISSING: ('gdp', 'resnet50', 'late')
MISSING: ('gdp', 'resnet50', 'gated')
MISSING: ('gdp', 'dinov2_vitb14', 'late')
MISSING: ('gdp', 'dinov2_vitb14', 'gated')
MISSING: ('acc2health', 'clip_vitb16', 'late')
MISSING: ('acc2health', 'clip_vitb16', 'gated')
MISSING: ('acc2health', 'resnet50', 'late')
MISSING: ('acc2health', 'resnet50', 'gated')
MISSING: ('acc2health', 'dinov2_vitb14', 'late')
MISSING: ('acc2health', 'dinov2_vitb14', 'gated')
MISSING: ('build_height', 'clip_vitb16', 'late')
MISSING: ('build_height', 'clip_vitb16', 'gated')
MISSING: ('build_height', 'resnet50', 'late')
MISSING: ('build_height', 'resnet50', 'gated')
MISSING: ('build_height', 'dinov2_vitb14', 'late')
MISSING: ('build_height', 'dinov2_vitb14', 'gated')
MISSING: ('pop', 'clip_vitb16', 'late')
MISSING: ('pop', 'clip_vitb16', 'gated')
MISSING: ('pop', 'resnet50', 'late')
MISSING: ('pop', 'resnet50', 'gated')
MIS

In [3]:
def find_metrics(task_name: str, satellite_model: str, epochs: int):
    task_root = RESULTS_ROOT / task_name
    if not task_root.exists():
        return None
    pattern = f"{task_name}-satellite-{satellite_model}-clip_vitb16-bs8-ep{epochs}-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42"
    exp_dir = task_root / pattern
    metrics_path = exp_dir / 'metrics.json'
    if not metrics_path.exists():
        return None
    payload = json.loads(metrics_path.read_text())
    return {
        'task': task_name,
        'model': satellite_model,
        'epochs': epochs,
        'best_epoch': payload.get('best_epoch'),
        'r2': payload.get('r2'),
        'rmse': payload.get('rmse'),
        'mae': payload.get('mae'),
        'mse': payload.get('mse'),
        'path': str(exp_dir),
    }


rows = []
for task_name, ref in LOCKED_PRITHVI.items():
    rows.append({
        'task': task_name,
        'model': 'prithvi_rgb_lora',
        'epochs': ref['epochs'],
        'best_epoch': ref['best_epoch'],
        'r2': ref['r2'],
        'rmse': ref['rmse'],
        'mae': None,
        'mse': None,
        'path': 'docs/PRITHVI_SATELLITE_REFERENCE.md',
    })

for task_name in TASK_EPOCHS:
    for satellite_model in ['dinov2_sat', 'resnet50_sat']:
        row = find_metrics(task_name, satellite_model, TASK_EPOCHS[task_name])
        if row is not None:
            rows.append(row)

df = pd.DataFrame(rows)
df.sort_values(['task', 'model']) if not df.empty else df

NameError: name 'LOCKED_PRITHVI' is not defined

In [9]:
if df.empty:
    print('No baseline comparison rows found yet.')
else:
    pivot = df.pivot_table(index='task', columns='model', values='r2', aggfunc='first')
    display(pivot)

    wins = 0
    considered = 0
    for task_name in ['build_height', 'gdp', 'acc2health']:
        if task_name not in pivot.index:
            continue
        if 'prithvi_rgb_lora' not in pivot.columns:
            continue
        other_values = [pivot.loc[task_name, c] for c in pivot.columns if c != 'prithvi_rgb_lora' and pd.notna(pivot.loc[task_name, c])]
        if not other_values:
            continue
        considered += 1
        if pivot.loc[task_name, 'prithvi_rgb_lora'] > max(other_values):
            wins += 1

    if considered == 0:
        print('Need baseline runs before making the modality decision.')
    elif wins >= 2:
        print('Recommendation: proceed to street-only clip_vitb16 and late fusion next.')
    else:
        print('Recommendation: stop and frame this as a smaller satellite empirical study unless stronger baseline results arrive.')


KeyError: 'model'

In [5]:
from pathlib import Path
import json
import pandas as pd

root = Path("/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned")

tasks = ["gdp", "acc2health", "build_height", "pop"]
models = ["prithvi_rgb_lora", "dinov2_sat", "resnet50_sat"]

# Epoch budgets you used
task_epochs = {"gdp": 20, "acc2health": 30, "build_height": 30, "pop": 5}

rows = []
for t in tasks:
    for m in models:
        exp = root / t / f"{t}-satellite-{m}-clip_vitb16-bs8-ep{task_epochs[t]}-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42"
        mp = exp / "metrics.json"
        if mp.exists():
            d = json.loads(mp.read_text())
            rows.append({
                "task": t, "model": m,
                "best_epoch": d.get("best_epoch"),
                "r2": d.get("r2"),
                "rmse": d.get("rmse"),
                "mae": d.get("mae"),
                "path": str(exp)
            })
        else:
            rows.append({
                "task": t, "model": m,
                "best_epoch": None, "r2": None, "rmse": None, "mae": None,
                "path": f"MISSING: {exp}"
            })

df = pd.DataFrame(rows)
print(df.sort_values(["task","model"]).to_string(index=False))

print("\nR2 Pivot:")
print(df.pivot_table(index="task", columns="model", values="r2", aggfunc="first"))

        task            model  best_epoch        r2         rmse          mae                                                                                                                                                                                                    path
  acc2health       dinov2_sat          20  0.098492 1.161133e+01 8.774560e+00           /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/acc2health/acc2health-satellite-dinov2_sat-clip_vitb16-bs8-ep30-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42
  acc2health prithvi_rgb_lora           9  0.390141 9.550184e+00 7.191029e+00     /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/acc2health/acc2health-satellite-prithvi_rgb_lora-clip_vitb16-bs8-ep30-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42
  acc2health     resnet50_sat          22  0.212407 1.085295e+01 7.204183e+00         /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learn

## Next-phase result summary (street + fusion)

Use this after running any of:

- street-only matrix
- fusion matrix

It scans `Results/global_learned/<task>/` and summarizes the best available runs for each branch/model combination under the same hyperparameter naming convention.

In [6]:
from pathlib import Path
import json
import pandas as pd

RESULTS_ROOT = DATA_ROOT / 'Results' / 'global_learned'
TASKS = ['gdp', 'acc2health', 'build_height', 'pop']


def load_metrics(exp_dir: Path):
    mp = exp_dir / 'metrics.json'
    if not mp.exists():
        return None
    payload = json.loads(mp.read_text())
    return {
        'best_epoch': payload.get('best_epoch'),
        'r2': payload.get('r2'),
        'rmse': payload.get('rmse'),
        'mae': payload.get('mae'),
        'mse': payload.get('mse'),
    }


def collect_branch_rows(task_name: str, branch: str):
    task_root = RESULTS_ROOT / task_name
    if not task_root.exists():
        return []
    rows = []
    for exp_dir in sorted([p for p in task_root.iterdir() if p.is_dir()]):
        name = exp_dir.name
        if f"{task_name}-{branch}-" not in name:
            continue
        metrics = load_metrics(exp_dir)
        if metrics is None:
            continue
        rows.append({
            'task': task_name,
            'branch': branch,
            'experiment': name,
            **metrics,
            'path': str(exp_dir),
        })
    return rows


all_rows = []
for t in TASKS:
    all_rows.extend(collect_branch_rows(t, 'satellite'))
    all_rows.extend(collect_branch_rows(t, 'street'))
    all_rows.extend(collect_branch_rows(t, 'fusion'))

summary_df = pd.DataFrame(all_rows)

if summary_df.empty:
    print('No metrics found yet for satellite/street/fusion in selected tasks.')
else:
    display(summary_df.sort_values(['task', 'branch', 'r2'], ascending=[True, True, False]).head(200))

    best_per_task_branch = (
        summary_df
        .sort_values(['task', 'branch', 'r2'], ascending=[True, True, False])
        .groupby(['task', 'branch'], as_index=False)
        .first()
    )

    print('\nBest run per task and branch:')
    display(best_per_task_branch[['task', 'branch', 'experiment', 'best_epoch', 'r2', 'rmse', 'mae']])

    print('\nR2 pivot (best per task/branch):')
    display(best_per_task_branch.pivot(index='task', columns='branch', values='r2'))

,task,branch,experiment,best_epoch,r2,rmse,mae,mse,path
24,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-resnet50-la...,12,0.190886,9.845183e+00,7.096077e+00,9.692764e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
21,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-clip_vitb16...,19,0.167151,9.988541e+00,6.845116e+00,9.977094e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
23,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-resnet50-ga...,3,0.161168,1.002436e+01,6.793436e+00,1.004878e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
22,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-clip_vitb16...,13,0.134735,1.018108e+01,6.703846e+00,1.036543e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
15,acc2health,satellite,acc2health-satellite-prithvi_rgb_lora-clip_vit...,9,0.390141,9.550184e+00,7.191029e+00,9.120602e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
16,acc2health,satellite,acc2health-satellite-prithvi_rgb_lora-clip_vit...,5,0.303444,1.020646e+01,7.174435e+00,1.041718e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
17,acc2health,satellite,acc2health-satellite-resnet50_sat-clip_vitb16-...,22,0.212407,1.085295e+01,7.204183e+00,1.177866e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
14,acc2health,satellite,acc2health-satellite-dinov2_sat-clip_vitb16-bs...,20,0.098492,1.161133e+01,8.774560e+00,1.348230e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
20,acc2health,street,acc2health-street-prithvi_rgb_lora-resnet50-me...,21,0.329908,8.959551e+00,6.307641e+00,8.027356e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
19,acc2health,street,acc2health-street-prithvi_rgb_lora-dinov2_vitb...,11,0.009981,1.089032e+01,7.447273e+00,1.185991e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...



Best run per task and branch:


,task,branch,experiment,best_epoch,r2,rmse,mae
0,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-resnet50-la...,12,0.190886,9.845183e+00,7.096077e+00
1,acc2health,satellite,acc2health-satellite-prithvi_rgb_lora-clip_vit...,9,0.390141,9.550184e+00,7.191029e+00
2,acc2health,street,acc2health-street-prithvi_rgb_lora-resnet50-me...,21,0.329908,8.959551e+00,6.307641e+00
3,build_height,fusion,build_height-fusion-prithvi_rgb_lora-dinov2_vi...,23,0.872343,2.527444e+00,1.878616e+00
4,build_height,satellite,build_height-satellite-prithvi_rgb_lora-clip_v...,11,0.859866,2.612972e+00,1.909996e+00
5,build_height,street,build_height-street-prithvi_rgb_lora-resnet50-...,10,0.344813,5.725870e+00,4.524409e+00
6,gdp,fusion,gdp-fusion-prithvi_rgb_lora-resnet50-gated-mea...,20,0.443721,4.230626e+08,2.311355e+08
7,gdp,satellite,gdp-satellite-prithvi_rgb_lora-clip_vitb16-bs8...,14,0.580773,3.314636e+08,1.981141e+08
8,gdp,street,gdp-street-prithvi_rgb_lora-resnet50-mean-bs8-...,7,0.362859,4.527680e+08,3.087936e+08
9,pop,fusion,pop-fusion-prithvi_rgb_lora-resnet50-gated-mea...,4,-0.294189,1.408040e+04,9.459062e+03



R2 pivot (best per task/branch):


branch,fusion,satellite,street
task,,,
acc2health,0.190886,0.390141,0.329908
build_height,0.872343,0.859866,0.344813
gdp,0.443721,0.580773,0.362859
pop,-0.294189,-0.032393,0.005767


In [7]:
from pathlib import Path
import json
import pandas as pd

RESULTS_ROOT = Path("/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned")
TASKS = ["gdp", "acc2health", "build_height", "pop"]
BRANCHES = ["satellite", "street", "fusion"]

def load_metrics(exp_dir: Path):
    mp = exp_dir / "metrics.json"
    if not mp.exists():
        return None
    d = json.loads(mp.read_text())
    return {
        "best_epoch": d.get("best_epoch"),
        "r2": d.get("r2"),
        "rmse": d.get("rmse"),
        "mae": d.get("mae"),
        "mse": d.get("mse"),
    }

rows = []
for t in TASKS:
    task_root = RESULTS_ROOT / t
    if not task_root.exists():
        continue
    for exp in task_root.iterdir():
        if not exp.is_dir():
            continue
        name = exp.name
        branch = None
        for b in BRANCHES:
            if f"{t}-{b}-" in name:
                branch = b
                break
        if branch is None:
            continue
        m = load_metrics(exp)
        if m is None:
            continue
        rows.append({
            "task": t,
            "branch": branch,
            "experiment": name,
            **m,
            "path": str(exp),
        })

df = pd.DataFrame(rows)
if df.empty:
    print("No run metrics found.")
else:
    df = df.sort_values(["task", "branch", "r2"], ascending=[True, True, False])
    display(df)

    best = df.groupby(["task", "branch"], as_index=False).first()
    print("\nBest run per task/branch:")
    display(best[["task","branch","experiment","best_epoch","r2","rmse","mae"]])

    print("\nR2 pivot (best per task/branch):")
    display(best.pivot(index="task", columns="branch", values="r2"))

    # optional: best model token parsed from experiment name
    def parse_model(exp_name, task, branch):
        p = exp_name.replace(f"{task}-{branch}-", "")
        # rough extraction before "-mean-" or "-clip_vitb16-" naming chunk
        return p.split("-mean-")[0].split("-clip_vitb16-")[0]
    best["model_guess"] = best.apply(lambda r: parse_model(r["experiment"], r["task"], r["branch"]), axis=1)
    print("\nBest model guess per task/branch:")
    display(best[["task","branch","model_guess","r2"]])

,task,branch,experiment,best_epoch,r2,rmse,mae,mse,path
23,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-resnet50-la...,12,0.190886,9.845183e+00,7.096077e+00,9.692764e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
22,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-clip_vitb16...,19,0.167151,9.988541e+00,6.845116e+00,9.977094e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
24,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-resnet50-ga...,3,0.161168,1.002436e+01,6.793436e+00,1.004878e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
21,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-clip_vitb16...,13,0.134735,1.018108e+01,6.703846e+00,1.036543e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
15,acc2health,satellite,acc2health-satellite-prithvi_rgb_lora-clip_vit...,9,0.390141,9.550184e+00,7.191029e+00,9.120602e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
14,acc2health,satellite,acc2health-satellite-prithvi_rgb_lora-clip_vit...,5,0.303444,1.020646e+01,7.174435e+00,1.041718e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
17,acc2health,satellite,acc2health-satellite-resnet50_sat-clip_vitb16-...,22,0.212407,1.085295e+01,7.204183e+00,1.177866e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
16,acc2health,satellite,acc2health-satellite-dinov2_sat-clip_vitb16-bs...,20,0.098492,1.161133e+01,8.774560e+00,1.348230e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...
19,acc2health,street,acc2health-street-prithvi_rgb_lora-resnet50-me...,21,0.329908,8.959551e+00,6.307641e+00,8.027356e+01,/content/drive/MyDrive/GeoBench/CityLens-Data/...
20,acc2health,street,acc2health-street-prithvi_rgb_lora-dinov2_vitb...,11,0.009981,1.089032e+01,7.447273e+00,1.185991e+02,/content/drive/MyDrive/GeoBench/CityLens-Data/...



Best run per task/branch:


,task,branch,experiment,best_epoch,r2,rmse,mae
0,acc2health,fusion,acc2health-fusion-prithvi_rgb_lora-resnet50-la...,12,0.190886,9.845183e+00,7.096077e+00
1,acc2health,satellite,acc2health-satellite-prithvi_rgb_lora-clip_vit...,9,0.390141,9.550184e+00,7.191029e+00
2,acc2health,street,acc2health-street-prithvi_rgb_lora-resnet50-me...,21,0.329908,8.959551e+00,6.307641e+00
3,build_height,fusion,build_height-fusion-prithvi_rgb_lora-dinov2_vi...,23,0.872343,2.527444e+00,1.878616e+00
4,build_height,satellite,build_height-satellite-prithvi_rgb_lora-clip_v...,11,0.859866,2.612972e+00,1.909996e+00
5,build_height,street,build_height-street-prithvi_rgb_lora-resnet50-...,10,0.344813,5.725870e+00,4.524409e+00
6,gdp,fusion,gdp-fusion-prithvi_rgb_lora-resnet50-gated-mea...,20,0.443721,4.230626e+08,2.311355e+08
7,gdp,satellite,gdp-satellite-prithvi_rgb_lora-clip_vitb16-bs8...,14,0.580773,3.314636e+08,1.981141e+08
8,gdp,street,gdp-street-prithvi_rgb_lora-resnet50-mean-bs8-...,7,0.362859,4.527680e+08,3.087936e+08
9,pop,fusion,pop-fusion-prithvi_rgb_lora-resnet50-gated-mea...,4,-0.294189,1.408040e+04,9.459062e+03



R2 pivot (best per task/branch):


branch,fusion,satellite,street
task,,,
acc2health,0.190886,0.390141,0.329908
build_height,0.872343,0.859866,0.344813
gdp,0.443721,0.580773,0.362859
pop,-0.294189,-0.032393,0.005767



Best model guess per task/branch:


,task,branch,model_guess,r2
0,acc2health,fusion,prithvi_rgb_lora-resnet50-late,0.190886
1,acc2health,satellite,prithvi_rgb_lora,0.390141
2,acc2health,street,prithvi_rgb_lora-resnet50,0.329908
3,build_height,fusion,prithvi_rgb_lora-dinov2_vitb14-late,0.872343
4,build_height,satellite,prithvi_rgb_lora,0.859866
5,build_height,street,prithvi_rgb_lora-resnet50,0.344813
6,gdp,fusion,prithvi_rgb_lora-resnet50-gated,0.443721
7,gdp,satellite,prithvi_rgb_lora,0.580773
8,gdp,street,prithvi_rgb_lora-resnet50,0.362859
9,pop,fusion,prithvi_rgb_lora-resnet50-gated,-0.294189


In [8]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1UuodhkBsWojV0Pe-we7ap9bLZMQFvVl19TUCBh754P0/edit#gid=0
